In [9]:
# from faster_whisper import WhisperModel, BatchedInferencePipeline
#
# stt = WhisperModel("tiny", device="cpu", compute_type="int8")
# stt_2 = WhisperModel("base", device="cpu", compute_type="int8")
# stt_3 = WhisperModel("small", device="cpu", compute_type="int8")

In [11]:
from utils_shiprocket import transcribe_item, prepare_data

In [12]:
train_df, val_df , _ = prepare_data(train_examples=None)

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/82 [00:00<?, ?it/s]

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "/Users/akshat.khatri/PycharmProjects/Shiprocket_final/kaggle/working/muril-endpoint-clf/checkpoint-3500"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()

def predict(texts, batch_size=32):
    preds, probs = [], []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors="pt").to(device)
            logits = model(**enc).logits
            p = torch.softmax(logits, dim=-1)
            preds.extend(torch.argmax(p, dim=-1).cpu().tolist())
            probs.extend(p[:, 1].cpu().tolist())
    return [{"text": t, "pred_endpoint_bool": bool(pr), "prob_true": pb} for t, pr, pb in zip(texts, preds, probs)]

if __name__ == "__main__":
    texts = ["How do I fix a running toilet?", "my name is akshat and"]
    for r in predict(texts):
        print(r)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'text': 'How do I fix a running toilet?', 'pred_endpoint_bool': True, 'prob_true': 0.9850984811782837}
{'text': 'my name is akshat and', 'pred_endpoint_bool': False, 'prob_true': 0.005232042632997036}
